# Qwen2.5-3B LoRA Fine-tune (Google Colab)

Kullanım:
1. Runtime → Change runtime type → **T4 GPU** seç.
2. Hücreleri sırayla çalıştır.
3. 2. hücrede `train.jsonl` ve `validation.jsonl` dosyalarını sorulduğunda yükle.
4. Son hücre `adapter.zip` olarak indirir (LoRA ağırlıkları + tokenizer).

Not: Bu bir **LoRA** eğitimi — tam model değil, küçük adaptör dosyaları üretir (Colab T4'te rahat sığar). Baz model + adaptör daha sonra birleştirilebilir.

In [ ]:
# 1) Kurulum (~2-3 dk)
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes
import torch
from google.colab import files
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK — runtime type kontrol et!')

In [ ]:
# 2) Veriyi yükle — train.jsonl ve validation.jsonl'ı seç
import os
from google.colab import files
up = files.upload()
os.makedirs('data', exist_ok=True)
for name, content in up.items():
    with open(f'data/{name}', 'wb') as f:
        f.write(content)
print('Yüklenenler:', list(up.keys()))

In [ ]:
# 3) Veriyi chat formatına çevir
from datasets import load_dataset
ds = load_dataset('json', data_files={'train': 'data/train.jsonl', 'validation': 'data/validation.jsonl'})
def fmt(ex):
    return {'text': (
        '<|im_start|>user\n' + ex['user'] + '<|im_end|>\n'
        '<|im_start|>assistant\n' + ex['assistant'] + '<|im_end|>'
    )}
ds = ds.map(fmt)
print(ds)
print(ds['train'][0]['text'][:300])

In [ ]:
# 4) Modeli yükle (4-bit, T4'e rahat sığar)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto')
model.config.use_cache = False
print('Model yüklendi.')

In [ ]:
# 5) LoRA ayarları + eğitim (vast.ai'daki 286 adımın karşılığı)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model = prepare_model_for_kbit_training(model)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

cfg = SFTConfig(
    output_dir='ckpt',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,   # efektif batch 16
    num_train_epochs=1,              # 4568 örnek / 16 ≈ 285 adım  → vast.ai ile aynı
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy='steps', eval_steps=50,
    save_strategy='steps', save_steps=100,
    bf16=False, fp16=True,
    report_to='none',
    max_length=1024,
    dataset_text_field='text',
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds['train'], eval_dataset=ds['validation'], processing_class=tok)
trainer.train()   # T4'te ~35-45 dk sürer
trainer.save_model('final_adapter')
tok.save_pretrained('final_adapter')
print('BİTTİ — final_adapter klasörüne kaydedildi.')

In [ ]:
# 6) Hızlı deneme (isteğe bağlı)
from transformers import pipeline
pipe = pipeline('text-generation', model=trainer.model, tokenizer=tok)
q = 'How do I say no without burning the bridge?'
prompt = f'<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n'
print(pipe(prompt, max_new_tokens=200, do_sample=False)[0]['generated_text'][len(prompt):])

In [ ]:
# 7) Sonucu bilgisayarına indir → qwen_adapter.zip
import shutil
from google.colab import files
shutil.make_archive('adapter', 'zip', 'final_adapter')
files.download('adapter.zip')